# Polinom Regresyon — Sağlık Sigortası Maliyeti**Veri:** insurance.csv (1.338 poliçe, 7 değişken)**Hedef:** `charges` — yıllık sigorta maliyeti

## 2. Kütüphaneler

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import PolynomialFeatures, StandardScalerfrom sklearn.linear_model import LinearRegressionfrom sklearn.pipeline import make_pipelinefrom sklearn.model_selection import train_test_split, cross_val_score, KFoldfrom sklearn.metrics import r2_score, mean_squared_errorimport warningswarnings.filterwarnings('ignore')# ── Grafik stili ayarları ──────────────────────────────────────────────────plt.rcParams['figure.figsize'] = (13, 5)plt.rcParams['figure.dpi'] = 110plt.rcParams['axes.spines.top'] = Falseplt.rcParams['axes.spines.right'] = Falsesns.set_style('whitegrid')sns.set_palette('husl')# ── Numpy yazdırma hassasiyeti ────────────────────────────────────────────np.set_printoptions(precision=4, suppress=True)

## 3. Veri Yükleme ve İlk İnceleme| Sütun | Açıklama ||---|---|| `age` | Yaş || `sex` | Cinsiyet || `bmi` | Vücut kitle indeksi (kg/m²) || `children` | Bakmakla yükümlü çocuk sayısı || `smoker` | Sigara kullanımı || `region` | Bölge || `charges` | Yıllık maliyet (USD) — hedef |

In [ ]:
# ── Veri setini oku ───────────────────────────────────────────────────────df = pd.read_csv('insurance.csv')print('=' * 60)print('📊 SAĞLIK SİGORTASI MALİYET VERİSİ')print('=' * 60)print(f'  Satır sayısı   : {df.shape[0]:,}')print(f'  Sütun sayısı   : {df.shape[1]}')print(f'  Yaş aralığı    : {df["age"].min()} → {df["age"].max()}')print(f'  BMI aralığı    : {df["bmi"].min():.2f} → {df["bmi"].max():.2f}')print(f'  Maliyet aralığı: ${df["charges"].min():,.2f} → ${df["charges"].max():,.2f}')print(f'  Sigara içen    : {(df["smoker"] == "yes").sum()} kişi (%{(df["smoker"] == "yes").mean()*100:.1f})')print()print('İlk 5 satır:')display(df.head())

### 3.1 Veri Tipleri ve Eksik Değer Analizi

In [ ]:
# ── Eksik değer analizi ───────────────────────────────────────────────────null_counts = df.isnull().sum()null_pct = (null_counts / len(df) * 100).round(2)null_df = pd.DataFrame({    'Eksik Sayı': null_counts,    'Eksik Yüzde (%)': null_pct,    'Veri Tipi': df.dtypes})print('📋 Eksik Değer ve Tip Raporu:')print(null_df.to_string())print()if null_counts.sum() == 0:    print('✅ Hiçbir sütunda eksik veri yok — imputation gerekmiyor.')else:    print(f'⚠️  Toplam {(null_counts > 0).sum()} sütunda eksik veri mevcut.')

### 3.2 İstatistiksel Özet

In [ ]:
# ── İstatistiksel özet ────────────────────────────────────────────────────stats = df[['age', 'bmi', 'children', 'charges']].describe().round(3)print('📈 İstatistiksel Özet:')print(stats)print()print('💡 İçgörüler:')print(f'  • Ortalama yaş      : {stats.loc["mean", "age"]:.1f}')print(f'  • Ortalama BMI      : {stats.loc["mean", "bmi"]:.2f} kg/m² (obezite sınırı 30)')print(f'  • Ortalama maliyet  : ${stats.loc["mean", "charges"]:,.0f}')print(f'  • Medyan maliyet    : ${stats.loc["50%", "charges"]:,.0f}')print(f'  • Maliyet std       : ${stats.loc["std", "charges"]:,.0f}')print()print('⚠️  Ortalama (${:,.0f}) medyandan (${:,.0f}) çok yüksek →'.format(    stats.loc["mean", "charges"], stats.loc["50%", "charges"]))print('    Maliyet dağılımı SAĞA ÇARPIK. Az sayıda çok pahalı poliçe var.')

## 4. Veri Ön İşlemeDuplike kontrolü ve IQR yöntemiyle aykırı değer incelemesi yapılıyor.

In [ ]:
# ── Duplike kontrolü ──────────────────────────────────────────────────────n_before = len(df)n_dup = df.duplicated().sum()print(f'🔁 Tekrar eden satır sayısı: {n_dup}')df = df.drop_duplicates().reset_index(drop=True)print(f'   Temizlik: {n_before} → {len(df)} satır')print()# ── IQR ile aykırı değer analizi ──────────────────────────────────────────print('📐 IQR Aykırı Değer Raporu:')for col in ['age', 'bmi', 'charges']:    Q1, Q3 = df[col].quantile([0.25, 0.75])    IQR = Q3 - Q1    alt, ust = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR    aykiri = ((df[col] < alt) | (df[col] > ust)).sum()    print(f'  {col:<10}: IQR={IQR:>9.2f} | sınırlar=[{alt:>10.2f}, {ust:>10.2f}] | aykırı={aykiri:>4} (%{aykiri/len(df)*100:.1f})')print()print('💡 KARAR: charges sütunundaki aykırı değerler SİLİNMEYECEK.')print('   Bunlar hatalı kayıt değil, sigara içen ve/veya obez bireylerin')print('   gerçek yüksek maliyetleridir. Silmek, modellemek istediğimiz')print('   olgunun ta kendisini veri setinden atmak olurdu.')

## 5. Keşifsel Veri Analizi### 5.1 Değişken Dağılımları

In [ ]:
# ── Dağılım grafikleri ────────────────────────────────────────────────────fig, axes = plt.subplots(1, 3, figsize=(16, 4))fig.suptitle('Sayısal Değişkenlerin Dağılımı', fontsize=14, fontweight='bold', y=1.02)vars_info = [    ('age',     '#3498db', 'Yaş'),    ('bmi',     '#e67e22', 'BMI (kg/m²)'),    ('charges', '#e74c3c', 'Sigorta Maliyeti ($)'),]for ax, (col, color, label) in zip(axes, vars_info):    sns.histplot(df[col], bins=40, kde=True, color=color, alpha=0.6, ax=ax)    mean_val, median_val = df[col].mean(), df[col].median()    ax.axvline(mean_val,   color='black', linestyle='-',  linewidth=1.5, label=f'Ort: {mean_val:.1f}')    ax.axvline(median_val, color='navy',  linestyle='--', linewidth=1.5, label=f'Med: {median_val:.1f}')    ax.set_xlabel(label)    ax.set_ylabel('Frekans')    ax.legend(fontsize=8)plt.tight_layout()plt.show()print('📌 age  : neredeyse düzgün (uniform) dağılmış')print('📌 bmi  : normale yakın, hafif sağa çarpık')print('📌 charges: güçlü sağa çarpık — ikinci bir tepe (bimodal yapı) göze çarpıyor')

### 5.2 Korelasyon Analizi

In [ ]:
# ── Korelasyon ısı haritası ve saçılım ────────────────────────────────────fig, axes = plt.subplots(1, 2, figsize=(15, 5))corr_matrix = df[['age', 'bmi', 'children', 'charges']].corr()mask = np.triu(np.ones_like(corr_matrix, dtype=bool))sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',            cmap='RdYlGn', center=0, vmin=-1, vmax=1,            square=True, linewidths=1, ax=axes[0],            annot_kws={'size': 12, 'weight': 'bold'})axes[0].set_title('Korelasyon Matrisi', fontweight='bold', pad=12)axes[1].scatter(df['bmi'], df['charges'], alpha=0.35, s=18, color='#8e44ad')axes[1].set_xlabel('BMI (kg/m²)')axes[1].set_ylabel('Maliyet ($)')axes[1].set_title(f'BMI → Maliyet  (r = {df["bmi"].corr(df["charges"]):.3f})', fontweight='bold')plt.tight_layout()plt.show()print(f'📉 BMI - charges korelasyonu sadece r = {df["bmi"].corr(df["charges"]):.3f}')print('   Sağdaki grafikte veri iki ayrı buluta bölünmüş görünüyor.')print('   Bu, gizli bir grup değişkeninin varlığına işaret ediyor. →  Bölüm 5.4')

### 5.3 Box Plot

In [ ]:
# ── Box plot ──────────────────────────────────────────────────────────────fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))sns.boxplot(y=df['bmi'], color='#e67e22', width=0.4, ax=axes[0])axes[0].axhline(30, color='red', linestyle='--', linewidth=1.5, label='Obezite sınırı (30)')axes[0].set_title('BMI Dağılımı', fontweight='bold')axes[0].legend()sns.boxplot(y=df['charges'], color='#e74c3c', width=0.4, ax=axes[1])axes[1].set_title('Maliyet Dağılımı (tüm veri)', fontweight='bold')axes[1].set_ylabel('Maliyet ($)')sns.boxplot(x='smoker', y='charges', data=df, palette=['#2ecc71', '#e74c3c'], width=0.5, ax=axes[2])axes[2].set_title('Maliyet — Sigara Durumuna Göre', fontweight='bold')axes[2].set_xlabel('Sigara kullanımı')axes[2].set_ylabel('Maliyet ($)')plt.tight_layout()plt.show()ic = df[df.smoker == 'yes']['charges'].median()icme = df[df.smoker == 'no']['charges'].median()print(f'📌 Medyan maliyet — sigara içen : ${ic:,.0f}')print(f'📌 Medyan maliyet — içmeyen     : ${icme:,.0f}')print(f'📌 Fark: {ic/icme:.1f} kat')

### 5.4 Kritik Keşif: Sigara Değişkeni Veriyi İkiye BölüyorBölüm 5.2'de BMI ile maliyet arasında neredeyse hiç doğrusal ilişki bulunamadı (r ≈ 0.20). Veri aslında iki ayrı popülasyondan oluşuyor ve bunlar üst üste bindirilince ilişki görünmez hale geliyor.

In [ ]:
# ── Sigara durumuna göre ayrıştırılmış saçılım ────────────────────────────fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))# Sol: Tüm veri, sigara durumuna göre renklendirilmişfor durum, renk, etiket in [('no', '#2ecc71', 'İçmiyor'), ('yes', '#e74c3c', 'İçiyor')]:    alt = df[df.smoker == durum]    axes[0].scatter(alt['bmi'], alt['charges'], alpha=0.5, s=20, color=renk, label=etiket)axes[0].axvline(30, color='black', linestyle='--', linewidth=1.5, alpha=0.7)axes[0].set_xlabel('BMI'); axes[0].set_ylabel('Maliyet ($)')axes[0].set_title('Tüm Veri — Sigara Durumuna Göre', fontweight='bold')axes[0].legend()# Orta: Sadece içmeyenleralt = df[df.smoker == 'no']axes[1].scatter(alt['bmi'], alt['charges'], alpha=0.5, s=20, color='#2ecc71')axes[1].set_xlabel('BMI'); axes[1].set_ylabel('Maliyet ($)')axes[1].set_title(f'Sigara İçmeyenler  (r = {alt["bmi"].corr(alt["charges"]):.3f})', fontweight='bold')# Sağ: Sadece içenleralt = df[df.smoker == 'yes']axes[2].scatter(alt['bmi'], alt['charges'], alpha=0.6, s=22, color='#e74c3c')axes[2].axvline(30, color='black', linestyle='--', linewidth=2, label='BMI = 30')axes[2].set_xlabel('BMI'); axes[2].set_ylabel('Maliyet ($)')axes[2].set_title(f'Sigara İçenler  (r = {alt["bmi"].corr(alt["charges"]):.3f})', fontweight='bold')axes[2].legend()plt.tight_layout()plt.show()print('🔑 BULGU:')print(f'   Tüm veride    r = {df["bmi"].corr(df["charges"]):.3f}  → ilişki yok gibi')print(f'   İçmeyenlerde  r = {df[df.smoker=="no"]["bmi"].corr(df[df.smoker=="no"]["charges"]):.3f}  → hâlâ zayıf')print(f'   İçenlerde     r = {df[df.smoker=="yes"]["bmi"].corr(df[df.smoker=="yes"]["charges"]):.3f}  → ÇOK GÜÇLÜ')print()print('   Sağdaki grafikte BMI = 30 civarında belirgin bir SIÇRAMA var.')print('   Bu doğrusal bir yapı değil — polinom regresyon tam olarak burada devreye giriyor.')

## 6. Polinom Regresyon Modeli**Ana senaryo:** Sigara içen bireylerde `bmi` → `charges`. Bu alt grup, eğrisel yapının en net göründüğü yer.### 6.1 Train-Test Ayrımı

In [ ]:
# ── Alt grubu seç ve özellik/hedef belirle ───────────────────────────────data = df[df.smoker == 'yes'].reset_index(drop=True)X = data[['bmi']].values      # shape: (n, 1) — 2D olması gerekiyory = data['charges'].values    # shape: (n,)# ── Train-Test split ──────────────────────────────────────────────────────X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.20, random_state=42)print('📊 Veri Bölme Sonuçları:')print(f'  Alt grup    : sigara içenler')print(f'  Toplam veri : {len(X):,} örnek')print(f'  Eğitim seti : {len(X_train):,} örnek ({len(X_train)/len(X)*100:.0f}%)')print(f'  Test seti   : {len(X_test):,} örnek  ({len(X_test)/len(X)*100:.0f}%)')print()print(f'  BMI aralığı     : {X.min():.2f} → {X.max():.2f}')print(f'  Maliyet aralığı : ${y.min():,.0f} → ${y.max():,.0f}')

### 6.2 Pipeline: PolynomialFeatures + LinearRegression

### 6.3 Farklı Derecelerde Model Eğitimi

In [ ]:
# ── Model eğitimi döngüsü ─────────────────────────────────────────────────degrees = [1, 2, 3, 4, 5, 7, 10]models = {}results = []for degree in degrees:    model = make_pipeline(        PolynomialFeatures(degree=degree, include_bias=True),        LinearRegression()    )    model.fit(X_train, y_train)          # SADECE eğitim verisiyle    models[degree] = model    y_train_pred = model.predict(X_train)    y_test_pred = model.predict(X_test)    results.append({        'Derece': degree,        'Train R²': round(r2_score(y_train, y_train_pred), 4),        'Test R²': round(r2_score(y_test, y_test_pred), 4),        'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_train_pred)), 1),        'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 1),        'Fark (Train-Test R²)': round(r2_score(y_train, y_train_pred) - r2_score(y_test, y_test_pred), 4)    })results_df = pd.DataFrame(results)print('📊 Derecelere Göre Performans:')display(results_df)

### 6.4 Regresyon Eğrilerinin Görselleştirilmesi

In [ ]:
# ── Regresyon eğrileri ────────────────────────────────────────────────────fig, axes = plt.subplots(2, 4, figsize=(18, 9))axes = axes.flatten()X_line = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)palette = sns.color_palette('husl', len(degrees))for i, degree in enumerate(degrees):    ax = axes[i]    model = models[degree]    row = results_df[results_df['Derece'] == degree].iloc[0]    ax.scatter(X_train, y_train, alpha=0.30, s=18, color='gray', label='Eğitim')    ax.scatter(X_test, y_test, alpha=0.55, s=22, color='#2c3e50', label='Test')    ax.plot(X_line, model.predict(X_line), color=palette[i], linewidth=2.5)    ax.axvline(30, color='red', linestyle='--', linewidth=1, alpha=0.5)    ax.set_title(f'Derece {degree}  |  Test R² = {row["Test R²"]:.3f}',                 fontweight='bold', fontsize=11)    ax.set_xlabel('BMI'); ax.set_ylabel('Maliyet ($)')    ax.set_ylim(y.min() - 3000, y.max() + 3000)    if i == 0:        ax.legend(fontsize=8)# Son paneli boş bırakma — özet metinaxes[-1].axis('off')en_iyi = results_df.loc[results_df['Test R²'].idxmax()]axes[-1].text(0.05, 0.5,              f"EN İYİ DERECE: {int(en_iyi['Derece'])}\n\n"              f"Test R²  = {en_iyi['Test R²']:.4f}\n"              f"Train R² = {en_iyi['Train R²']:.4f}\n"              f"Test RMSE = ${en_iyi['Test RMSE']:,.0f}\n\n"              f"Kırmızı kesik çizgi:\nBMI = 30 (obezite sınırı)",              fontsize=12, va='center', family='monospace')plt.tight_layout()plt.show()

## 7. Model Performans Karşılaştırması

In [ ]:
# ── Performans görselleştirmesi ────────────────────────────────────────────fig, axes = plt.subplots(1, 3, figsize=(18, 5))fig.suptitle('Model Performans Karşılaştırması', fontsize=13, fontweight='bold')# 1. R² karşılaştırmasıaxes[0].plot(results_df['Derece'], results_df['Train R²'],             'o-', color='#2ecc71', linewidth=2.5, markersize=7, label='Train R²')axes[0].plot(results_df['Derece'], results_df['Test R²'],             's--', color='#e74c3c', linewidth=2.5, markersize=7, label='Test R²')axes[0].fill_between(results_df['Derece'], results_df['Train R²'], results_df['Test R²'],                     alpha=0.15, color='orange')axes[0].set_xlabel('Polinom Derecesi'); axes[0].set_ylabel('R²')axes[0].set_title('R² — Train vs Test', fontweight='bold')axes[0].legend(); axes[0].grid(alpha=0.3)# 2. RMSEaxes[1].plot(results_df['Derece'], results_df['Train RMSE'],             'o-', color='#2ecc71', linewidth=2.5, markersize=7, label='Train RMSE')axes[1].plot(results_df['Derece'], results_df['Test RMSE'],             's--', color='#e74c3c', linewidth=2.5, markersize=7, label='Test RMSE')axes[1].set_xlabel('Polinom Derecesi'); axes[1].set_ylabel('RMSE ($)')axes[1].set_title('RMSE — Train vs Test', fontweight='bold')axes[1].legend(); axes[1].grid(alpha=0.3)# 3. Overfitting göstergesiaxes[2].bar(results_df['Derece'].astype(str), results_df['Fark (Train-Test R²)'],            color=['#2ecc71' if f < 0.05 else '#e67e22' if f < 0.15 else '#e74c3c'                   for f in results_df['Fark (Train-Test R²)']])axes[2].axhline(0, color='black', linewidth=0.8)axes[2].set_xlabel('Polinom Derecesi'); axes[2].set_ylabel('Train R² − Test R²')axes[2].set_title('Overfitting Göstergesi', fontweight='bold')axes[2].grid(axis='y', alpha=0.3)plt.tight_layout()plt.show()print('💡 Fark büyüdükçe model eğitim verisini ezberliyor demektir.')

### 7.1 K-Fold Cross-Validation ile Derece SeçimiBu alt grupta 274 gözlem olduğu için CV, tek bölmeye göre daha güvenilir.

In [ ]:
# ── K-Fold Cross-Validation ────────────────────────────────────────────────print('🔄 5-Fold Cross-Validation Sonuçları:')print('─' * 55)kf = KFold(n_splits=5, shuffle=True, random_state=42)cv_results = []for degree in degrees:    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')    cv_results.append({        'Derece': degree,        'CV R² Ort': round(cv_scores.mean(), 4),        'CV R² Std': round(cv_scores.std(), 4),        'En Düşük Fold': round(cv_scores.min(), 4)    })cv_df = pd.DataFrame(cv_results)display(cv_df)en_iyi_cv = int(cv_df.loc[cv_df['CV R² Ort'].idxmax(), 'Derece'])print(f'\n🏆 CV skoruna göre en iyi derece: {en_iyi_cv}')print(f'   Doğrusal modele (derece 1) göre kazanç: '      f'{cv_df.loc[cv_df["Derece"]==en_iyi_cv, "CV R² Ort"].values[0] - cv_df.loc[cv_df["Derece"]==1, "CV R² Ort"].values[0]:+.4f} R²')print()print('⚠️  Yüksek derecelerdeki NEGATİF R² değerlerine dikkat:')print('   R² < 0 demek, modelin basit ortalama tahminden bile KÖTÜ olduğu anlamına gelir.')print('   Overfitting\'in en dramatik göstergesi budur.')

In [ ]:
# ── CV sonuçlarının görselleştirilmesi ────────────────────────────────────plt.figure(figsize=(11, 5))plt.errorbar(cv_df['Derece'], cv_df['CV R² Ort'], yerr=cv_df['CV R² Std'],             fmt='o-', capsize=5, capthick=2, linewidth=2.5, markersize=8,             color='#8e44ad', ecolor='#95a5a6', label='CV R² (± std)')plt.axhline(0, color='red', linestyle='--', linewidth=1.5, label='R² = 0 (ortalama tahmin seviyesi)')plt.axvline(en_iyi_cv, color='green', linestyle=':', linewidth=2, label=f'En iyi derece = {en_iyi_cv}')plt.xlabel('Polinom Derecesi'); plt.ylabel('Cross-Validation R²')plt.title('K-Fold CV ile Derece Seçimi', fontweight='bold', fontsize=13)plt.legend(); plt.grid(alpha=0.3)plt.show()

## 8. En İyi Model — Detaylı Analiz

In [ ]:
# ── En iyi modeli seç ─────────────────────────────────────────────────────best_degree = en_iyi_cvbest_model = make_pipeline(PolynomialFeatures(best_degree), LinearRegression())best_model.fit(X_train, y_train)y_test_pred = best_model.predict(X_test)y_train_pred = best_model.predict(X_train)residuals = y_test - y_test_predprint(f'🏆 Seçilen Model: {best_degree}. Derece Polinom')print(f'   Train R²   = {r2_score(y_train, y_train_pred):.5f}')print(f'   Test  R²   = {r2_score(y_test, y_test_pred):.5f}')print(f'   Test  RMSE = ${np.sqrt(mean_squared_error(y_test, y_test_pred)):,.2f}')print(f'   Test  MAE  = ${np.mean(np.abs(residuals)):,.2f}')print()print(f'   Karşılaştırma — doğrusal model (derece 1):')lin = models[1]print(f'   Test  R²   = {r2_score(y_test, lin.predict(X_test)):.5f}')print(f'   Test  RMSE = ${np.sqrt(mean_squared_error(y_test, lin.predict(X_test))):,.2f}')

In [ ]:
# ── Artık (residual) analizi ──────────────────────────────────────────────fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))axes[0].scatter(y_test_pred, residuals, alpha=0.6, s=35, color='#3498db')axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)axes[0].set_xlabel('Tahmin Edilen Maliyet ($)'); axes[0].set_ylabel('Artık ($)')axes[0].set_title('Artık Grafiği', fontweight='bold')axes[0].grid(alpha=0.3)sns.histplot(residuals, bins=20, kde=True, color='#9b59b6', alpha=0.7, ax=axes[1])axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)axes[1].set_xlabel('Artık ($)')axes[1].set_title('Artık Dağılımı (Normallik Kontrolü)', fontweight='bold')axes[2].scatter(y_test, y_test_pred, alpha=0.6, s=35, color='#e67e22')axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],             'r--', linewidth=2, label='Mükemmel tahmin (y=x)')axes[2].set_xlabel('Gerçek Maliyet ($)'); axes[2].set_ylabel('Tahmin Edilen ($)')axes[2].set_title('Gerçek vs Tahmin', fontweight='bold')axes[2].legend(); axes[2].grid(alpha=0.3)plt.tight_layout()plt.show()print(f'📐 Artıkların ortalaması : ${np.mean(residuals):,.2f}  (0\'a yakın olmalı)')print(f'📐 Artıkların std sapması: ${np.std(residuals):,.2f}')

## 9. Model Katsayıları

In [ ]:
# ── Katsayı çıkarımı ──────────────────────────────────────────────────────lr_step = best_model.named_steps['linearregression']pf_step = best_model.named_steps['polynomialfeatures']intercept = lr_step.intercept_coeffs = lr_step.coef_feat_names = pf_step.get_feature_names_out(['bmi'])print(f'📐 {best_degree}. Derece Polinom Katsayıları:')print('─' * 50)print(f'  β₀ (intercept): {intercept:,.4f}')print()for i, (name, coef) in enumerate(zip(feat_names, coeffs)):    print(f'  β{i} ({name:>8s}): {coef:>18,.6f}')print()print('📝 Model denklemi:')terimler = [f'{intercept:,.2f}']for name, coef in zip(feat_names[1:], coeffs[1:]):    terimler.append(f'({coef:,.4f} × {name})')print('  charges = ' + ' + '.join(terimler))

## 10. İkinci Senaryo: Yaş → Maliyet (Sigara İçmeyenler)

In [ ]:
# ── Sigara içmeyenlerde age → charges ────────────────────────────────────data2 = df[df.smoker == 'no'].reset_index(drop=True)X2 = data2[['age']].valuesy2 = data2['charges'].valuesX2_train, X2_test, y2_train, y2_test = train_test_split(    X2, y2, test_size=0.20, random_state=42)degrees2 = [1, 2, 3, 5, 7]models2 = {}results2 = []for degree in degrees2:    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())    model.fit(X2_train, y2_train)    models2[degree] = model    cv_r2 = cross_val_score(model, X2, y2, cv=kf, scoring='r2').mean()    results2.append({        'Derece': degree,        'Train R²': round(r2_score(y2_train, model.predict(X2_train)), 4),        'Test R²': round(r2_score(y2_test, model.predict(X2_test)), 4),        'CV R²': round(cv_r2, 4),        'Test RMSE': round(np.sqrt(mean_squared_error(y2_test, model.predict(X2_test))), 1)    })results2_df = pd.DataFrame(results2)print(f'📊 Senaryo 2 — Sigara içmeyenler (n = {len(data2)})')display(results2_df)

In [ ]:
# ── İkinci senaryonun görselleştirilmesi ──────────────────────────────────fig, axes = plt.subplots(1, 2, figsize=(15, 5))X2_line = np.linspace(X2.min(), X2.max(), 300).reshape(-1, 1)renkler = sns.color_palette('husl', len(degrees2))axes[0].scatter(X2, y2, alpha=0.35, s=20, color='gray', label='Veri')for i, degree in enumerate(degrees2):    axes[0].plot(X2_line, models2[degree].predict(X2_line),                 linewidth=2, color=renkler[i], label=f'Derece {degree}')axes[0].set_xlabel('Yaş'); axes[0].set_ylabel('Maliyet ($)')axes[0].set_title('Yaş → Maliyet (Sigara İçmeyenler)', fontweight='bold')axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)axes[1].plot(results2_df['Derece'], results2_df['Train R²'], 'o-',             color='#2ecc71', linewidth=2.5, markersize=8, label='Train R²')axes[1].plot(results2_df['Derece'], results2_df['Test R²'], 's--',             color='#e74c3c', linewidth=2.5, markersize=8, label='Test R²')axes[1].plot(results2_df['Derece'], results2_df['CV R²'], '^:',             color='#8e44ad', linewidth=2.5, markersize=8, label='CV R²')axes[1].set_xlabel('Polinom Derecesi'); axes[1].set_ylabel('R²')axes[1].set_title('Senaryo 2 — Derece Karşılaştırması', fontweight='bold')axes[1].legend(); axes[1].grid(alpha=0.3)plt.tight_layout()plt.show()fark = results2_df.loc[results2_df['Derece']==2, 'CV R²'].values[0] - results2_df.loc[results2_df['Derece']==1, 'CV R²'].values[0]print(f'📌 Derece 2 ile derece 1 arasındaki CV R² farkı: {fark:+.4f}')print('   Bu fark ihmal edilebilir düzeyde. Yaş-maliyet ilişkisi neredeyse tam doğrusal.')print('   SONUÇ: Bu senaryoda polinom regresyona GEREK YOK — basit doğrusal model yeterli.')print('   Grafikte görünen dağınıklık eğrisellikten değil, düşük/yüksek maliyetli')print('   iki ayrı banttan kaynaklanıyor (kronik hastalığı olanlar ve olmayanlar).')

## 11. Bulgular| Konu | Bulgu ||------|-------|| BMI–charges (tüm veri) | r ≈ 0.20 — ilişki yok gibi görünüyor || Karıştırıcı değişken | `smoker` veriyi iki popülasyona bölüyor || BMI–charges (sigara içenler) | r ≈ 0.81, BMI=30'da sıçramalı || En iyi derece (Senaryo 1) | 3–4 || Derece 10 | CV R² negatife düşüyor || Senaryo 2 (yaş → maliyet) | Polinom terimleri katkı sağlamıyor |**Derece taraması (sigara içenler, CV R²):**| Derece | CV R² ||---|---|| 1 | 0.642 || 3 | 0.687 || 4 | 0.697 || 5 | 0.209 || 10 | −0.570 |Derece 10'da R² negatife düşüyor, yani model basit ortalama tahminden bile kötü hale geliyor.**Metodolojik ders:** Polinom regresyon doğrudan tüm veriye uygulansaydı hiçbir derece işe yaramayacak ve "bu veri modellenemiyor" sonucuna varılacaktı (tüm veride R² ≈ 0.03). Doğru adım önce veriyi anlamak, karıştırıcı değişkeni bulup alt grupları ayırmak, sonra model kurmaktı.**Ekstrapolasyon kısıtı:** Eğitim aralığının dışında model kontrolsüz tahminler üretiyor. BMI=70 için 238.000 dolar gibi anlamsız değerler çıkıyor.**Alternatif yaklaşım:** BMI = 30'daki sıçrama bir eşik etkisi. `bmi_obez = (bmi >= 30)` kukla değişkeni eklenmiş çoklu doğrusal regresyon, daha az parametreyle daha yorumlanabilir bir model verebilir. Polinom regresyon burada eğriselliği keşfetmek için doğru araç; nihai model için tek seçenek değil.

## 12. Tahmin Fonksiyonu

In [ ]:
# ── Tahmin fonksiyonu ─────────────────────────────────────────────────────def maliyet_tahmin(bmi_listesi, model=best_model, degree=best_degree):    # Sigara icen bireyler icin verilen BMI degerlerine gore maliyet tahmin eder.    #    # Parametreler:    #   bmi_listesi : list veya float - Tahmin yapilacak BMI degerleri    #   model       : Egitilmis pipeline (varsayilan: en iyi model)    #   degree      : Kullanilan polinom derecesi    #    # Dondurur:    #   pandas DataFrame - BMI, tahmini maliyet ve aralik disi uyarisi    if isinstance(bmi_listesi, (int, float)):        bmi_listesi = [bmi_listesi]    girdi = np.array(bmi_listesi).reshape(-1, 1)    tahminler = model.predict(girdi)    bmi_min, bmi_max = X.min(), X.max()    uyarilar = ['⚠️ ARALIK DIŞI' if (b < bmi_min or b > bmi_max) else 'OK'                for b in bmi_listesi]    return pd.DataFrame({        'BMI': bmi_listesi,        'Tahmini Maliyet ($)': np.round(tahminler, 2),        'Durum': uyarilar    })print(f'Model: {best_degree}. derece polinom | Eğitim aralığı: BMI {X.min():.1f} – {X.max():.1f}')print()display(maliyet_tahmin([20, 25, 28, 30, 32, 35, 40, 45]))print()print('🔬 Ekstrapolasyon testi — eğitim aralığının dışı:')display(maliyet_tahmin([10, 60, 70]))print()print('Yukarıdaki değerlere dikkat: BMI=10 için eğitim aralığındaki BMI=20\'den')print('DAHA YÜKSEK, BMI=70 için ise tamamen fantastik bir maliyet çıkıyor.')print('Eğri, veri bittiği anda kontrolsüz biçimde savruluyor.')print('Bu, polinom regresyonun en önemli kısıtıdır: ekstrapolasyon güvenilmezdir.')